# 12 · Combined area, checked against shapely

## What this checks

**Combined area** joins several areas into one and measures the ground they cover between them. It also reports the
sum of the separate areas, with their holes taken out. This check covers release 0.2.0.

Joining shapes is called a union. A union is where two geometry libraries most often differ. That is where they
decide edges that coincide and crossings that almost touch. So it is worth checking.

## How it is checked

This is one of the most independent checks in the lab. The two sides use different tools at every step:

- **Fieldwork** joins the shapes with polyclip-ts, a geometry library for the browser. It measures the result with the
  shoelace formula, the standard way to find an area from its corners.
- **We** join and measure with **shapely**, Python's link to GEOS. GEOS is the geometry engine much GIS software uses.

Nothing here uses Fieldwork's code.

The data are three corridors from the Broad Street pump, in the John Snow worked example: 2, 5 and 10 minutes. They
are **nested** on purpose. A shorter corridor lies inside a longer one. So the areas overlap heavily, and the ground
they cover is far less than their sum. With areas that did not overlap, the union would simply equal the sum, and the
check would test nothing.

## What should happen

1. **The same joined shape**, with at most 1 m² of ground in one shape but not the other.
2. **The same combined area**, within 1 m².
3. **The same sum of the separate areas**, within 1 m².
4. **The same number of areas in, and of pieces out.**

One square metre in about 660,000 is under two parts in a million. These criteria were fixed before the check was
run.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj", "shapely", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_12_combined_area import run

report = run(Path.cwd() / "fixtures" / "combined-area-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["parts"], "parts")
print(f"combined area   fieldwork {report['fieldwork']['combinedAreaSquareMetres']:12.2f}   "
      f"shapely {report['reference']['combinedAreaSquareMetres']:12.2f}")
print(f"parts, net      fieldwork {report['fieldwork']['partsAreaSquareMetres']:12.2f}   "
      f"shapely {report['reference']['partsAreaSquareMetres']:12.2f}")
print(f"pieces          fieldwork {report['fieldwork']['combinedPieceCount']:>12}   "
      f"shapely {report['reference']['combinedPieceCount']:>12}")
print()
print(f"symmetric difference of the union {report['geometryDifferenceSquareMetres']:.3e} m2")
print(f"polyclip-ts against GEOS: two unions, independently implemented.")

## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.2.0:
**Agrees**.

Over the 3 areas, both sides cover 659,510.5 m². They differ by 0.0052 m². The joined shapes match exactly: no
ground lies in one but not the other. Both give one piece. The separate areas sum to 847,861 m² on both sides, within
0.0043 m², so 188,351 m² is ground covered more than once.

When first recorded, the three covered 701,662.61 m². The corridors are smaller now, because Fieldwork charges the
walk from the pump onto the street network against the budget. The data changed; the agreement did not.

Two union methods, written separately, give the same ground and the same measure of it.

Running the cell above repeats the check on the same data.

## What this does not show

- That the combined area means what a reader will take it to mean. It is the ground covered by at least one corridor.
  It is not a population, not a catchment, and not a service area.